In [0]:
%python
import subprocess
import os
import requests

# Generate key
for f in ["/tmp/brev_key", "/tmp/brev_key.pub"]:
    if os.path.exists(f):
        os.remove(f)

subprocess.run('ssh-keygen -t ed25519 -f /tmp/brev_key -N "" -q', shell=True, check=True)
os.chmod("/tmp/brev_key", 0o600)

with open("/tmp/brev_key") as f:
    private_key = f.read()
with open("/tmp/brev_key.pub") as f:
    public_key = f.read()

# Store in Databricks secrets via API
host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().getOrElse(None)
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().getOrElse(None)
headers = {"Authorization": f"Bearer {token}"}

for key_name, key_value in [("ssh_private_key", private_key), ("ssh_public_key", public_key)]:
    resp = requests.post(
        f"{host}/api/2.0/secrets/put",
        headers=headers,
        json={
            "scope": "brev",
            "key": key_name,
            "string_value": key_value
        }
    )
    print(f"{key_name}: {resp.status_code}")

# Clean up
os.remove("/tmp/brev_key")
os.remove("/tmp/brev_key.pub")

print("Keys stored in secrets.")

In [0]:
# Brev Enironment Setup

# use -> databricks secrets put-secret brev token --string-value "YOUR_TOKEN"
# (https://docs.databricks.com/aws/en/security/secrets/?language=Databricks%C2%A0CLI)

# Get secrets from Databricks
# Three scopes: |brev:              | wandb:     | databricks: 
#               |  -token           |   -token   |   - pat (personal access token)
#               |  -instance        |            |
#               |  -ssh_public_key  |            |
#               |  -ssh_private_key |            |
#               |  -dataset_name    |            |
#               |  -max_steps       |            |
#               |  -save_steps      |            |

# print the value of a secret from a scope: databricks secrets get-secret <scope-name> <key-name> | jq -r .value | base64 --decode

brev_token = dbutils.secrets.get(scope="brev", key="token")
brev_instance = dbutils.secrets.get(scope="brev", key="instance")
dataset_name = dbutils.secrets.get(scope="brev", key="dataset_name")
max_steps = dbutils.secrets.get(scope="brev", key="max_steps")
save_steps = dbutils.secrets.get(scope="brev", key="save_steps")

ssh_pub_key = dbutils.secrets.get(scope="brev", key="ssh_public_key")
ssh_priv_key = dbutils.secrets.get(scope="brev", key="ssh_private_key")

wandb = dbutils.secrets.get(scope="wandb", key="token")

pat = dbutils.secrets.get(scope="databricks", key="pat") # pe cand da cristi token u

host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().getOrElse(None) # ca sa luam datasetu cu wget

import os
os.environ['BREV_TOKEN'] = brev_token
os.environ['BREV_INSTANCE_NAME'] = brev_instance

os.environ['SSH_PUB_KEY'] = ssh_pub_key
os.environ['SSH_PRIV_KEY'] = ssh_priv_key

os.environ['WANDB_API_KEY'] = wandb
os.environ['SAVE_STEPS'] = save_steps
os.environ['MAX_STEPS'] = max_steps

# Dataset
os.environ['DATASET_PATH'] = '/Volumes/workspace/default/datasets' 
os.environ['DATASET_NAME'] = dataset_name

# Modality files
os.environ['MODALITY_FILES_PATH'] = '/Volumes/workspace/default/modality_files'
os.environ['MODALITY_JSON'] = 'modality.json'
os.environ['MODALITY_PY'] = 'so100_top_wrist_config.py'
 
os.environ['DATABRICKS_TOKEN'] = pat
os.environ['DATABRICKS_HOST'] = host

In [0]:
%sh
set -euo pipefail # bash safety net

INSTALL_DIR="$HOME/.local/bin"
mkdir -p "$INSTALL_DIR"

# Download latest Brev CLI release matching this machine
OS="$(uname -s | tr '[:upper:]' '[:lower:]')" # ex: "linux"
ARCH="$(uname -m)"                            # ex: x86_64"

# normalization case to match the naming convention used in Brev's GitHub release assets
case "$ARCH" in                               
  x86_64) ARCH="amd64" ;;
  aarch64|arm64) ARCH="arm64" ;;
esac

# Download the latest Brev CLI
URL=$(curl -fsSL https://api.github.com/repos/brevdev/brev-cli/releases/latest | grep "browser_download_url.*${OS}.*${ARCH}" | cut -d '"' -f 4)

# is a guard if the URL is empty
test -n "$URL"

# Downloads and installs the Brev CLI binary to ~/.local/bin/ and adds it to PATH.
tmp="$(mktemp -d)"
curl -fsSL "$URL" -o "$tmp/brev.tgz"
tar -xzf "$tmp/brev.tgz" -C "$tmp"

mv "$tmp/brev" "$INSTALL_DIR/brev"
chmod +x "$INSTALL_DIR/brev"

export PATH="$INSTALL_DIR:$PATH"

echo "=== Checking Brev login ==="

echo "No" | brev login --token "$BREV_TOKEN" --skip-browser


In [0]:
%sh
export PATH="$HOME/.local/bin:$PATH"

brev ssh "$BREV_INSTANCE_NAME" << EOF
\$HOME/.ssh/authorized_keys 2>/dev/null || echo '$SSH_PUB_KEY' >> \$HOME/.ssh/authorized_keys
chmod 700 \$HOME/.ssh
chmod 600 \$HOME/.ssh/authorized_keys
echo 'KEY_INSTALLED'
cat \$HOME/.ssh/authorized_keys
EOF


In [0]:
%sh
export PATH="$HOME/.local/bin:$PATH"

brev ssh "$BREV_INSTANCE_NAME" << 'EOF' > /tmp/brev_ip.txt 2>/dev/null
curl -s ifconfig.me
EOF

BREV_IP=$(tail -1 /tmp/brev_ip.txt)
echo "Brev IP: $BREV_IP"


In [0]:
%python
import os
with open("/tmp/brev_ip.txt") as f:
    lines = f.read().strip().split("\n")
    os.environ["BREV_INSTANCE_IP"] = lines[-1]
print(f"IP: {os.environ['BREV_INSTANCE_IP']}")

resp = requests.post(
        f"{host}/api/2.0/secrets/put",
        headers=headers,
        json={
            "scope": "brev",
            "key": "brev_ip",
            "string_value": os.environ['BREV_INSTANCE_IP']
        }
    )
print(f"brev_ip: {resp.status_code}")
os.remove("/tmp/brev_ip.txt")